In [1]:
!pip install pymupdf


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz
import pandas as pd
import re
from pathlib import Path

In [3]:
pdf_path = "D:/sudhendra/learning projects/GraphRAG/data/Apple23 report.pdf"

doc = fitz.open(pdf_path)

len(doc)

80

In [5]:
all_lines = []

for page_index, page in enumerate(doc):
    words = page.get_text("words")

    line_groups = {}

    for word in words:
        x0, y0, x1, y1, text, block_no, line_no, word_no = word

        key = (block_no, line_no)

        if key not in line_groups:
            line_groups[key] = []

        line_groups[key].append((x0, text))

    for (block_no, line_no), line_words in line_groups.items():
        line_words = sorted(line_words, key=lambda x: x[0])

        line_text = " ".join([word for _, word in line_words])

        all_lines.append({
            "page": page_index + 1,
            "block": block_no,
            "line": line_no,
            "text": line_text
        })

lines_df = pd.DataFrame(all_lines)

lines_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5227 entries, 0 to 5226
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   page    5227 non-null   int64
 1   block   5227 non-null   int64
 2   line    5227 non-null   int64
 3   text    5227 non-null   str  
dtypes: int64(3), str(1)
memory usage: 163.5 KB


In [6]:
financial_lines_df = lines_df[
    lines_df["text"].str.contains(
        "net sales|gross margin|operating income|net income|research and development|total assets|total liabilities|cash and cash equivalents",
        case=False,
        regex=True,
        na=False
    )
].copy()

financial_lines_df[["page", "text"]].head(50)

,page,text
262,5,sales through its direct and indirect distribu...
265,5,competition and resulting downward pressure on...
300,6,Research and Development
304,6,"products and services, and to expand the range..."
324,7,The Company has historically experienced highe...
325,7,"part to seasonal holiday demand. Additionally,..."
326,7,sales and operating expenses. The timing of pr...
328,7,older product often declines as the launch of ...
387,8,The Company has international operations with ...
411,9,"The Company has a large, global business with ..."


In [7]:
def clean_number(value):
    value = str(value).strip()

    negative = False

    if "(" in value and ")" in value:
        negative = True

    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.replace("(", "")
    value = value.replace(")", "")
    value = value.strip()

    try:
        num = float(value)
        return -num if negative else num
    except:
        return None

In [8]:
def parse_financial_line(text):
    text = str(text)

    number_strings = re.findall(r"\(?\$?\s?\d[\d,]*\.?\d*\)?", text)

    numbers = []

    for n in number_strings:
        cleaned = clean_number(n)
        if cleaned is not None:
            numbers.append(cleaned)

    metric_name = re.sub(r"\(?\$?\s?\d[\d,]*\.?\d*\)?", "", text)
    metric_name = metric_name.replace("$", "")
    metric_name = re.sub(r"\s+", " ", metric_name).strip().lower()

    return metric_name, numbers

In [9]:
structured_rows = []

for _, row in financial_lines_df.iterrows():
    metric_name, numbers = parse_financial_line(row["text"])

    if len(numbers) >= 3:
        structured_rows.append({
            "page": int(row["page"]),
            "metric_name": metric_name,
            "2023": numbers[0],
            "2022": numbers[1],
            "2021": numbers[2],
            "raw_text": row["text"]
        })

metrics_df = pd.DataFrame(structured_rows)

metrics_df.head(30)

,page,metric_name,2023,2022,2021,raw_text
0,23,the company’s total net sales were billion and...,383.3,97.0,2023.0,The Company’s total net sales were $383.3 bill...
1,23,the company’s total net sales decreased% or bi...,3.0,11.0,2023.0,The Company’s total net sales decreased 3% or ...
2,24,the following table shows net sales by reporta...,2023.0,2022.0,2021.0,The following table shows net sales by reporta...
3,24,americas net sales decreased% or billion durin...,4.0,7.1,2023.0,Americas net sales decreased 4% or $7.1 billio...
4,24,europe net sales decreased% or million during ...,1.0,824.0,2023.0,Europe net sales decreased 1% or $824 million ...
5,24,greater china net sales decreased% or billion ...,2.0,1.6,2023.0,Greater China net sales decreased 2% or $1.6 b...
6,24,japan net sales decreased% or billion during c...,7.0,1.7,2023.0,Japan net sales decreased 7% or $1.7 billion d...
7,24,rest of asia pacific net sales increased% or m...,1.0,240.0,2023.0,Rest of Asia Pacific net sales increased 1% or...
8,25,the following table shows net sales by categor...,2023.0,2022.0,2021.0,The following table shows net sales by categor...
9,25,iphone net sales decreased% or billion during ...,2.0,4.9,2023.0,iPhone net sales decreased 2% or $4.9 billion ...


In [10]:
metrics_df[
    metrics_df["metric_name"].str.contains("total net sales", case=False, na=False)
]

,page,metric_name,2023,2022,2021,raw_text
0,23,the company’s total net sales were billion and...,383.3,97.0,2023.0,The Company’s total net sales were $383.3 bill...
1,23,the company’s total net sales decreased% or bi...,3.0,11.0,2023.0,The Company’s total net sales decreased 3% or ...
16,38,total net sales include billion of revenue rec...,8.2,2023.0,24.0,Total net sales include $8.2 billion of revenu...


In [11]:
metrics_df[
    metrics_df["metric_name"].str.contains("net income", case=False, na=False)
]

,page,metric_name,2023,2022,2021,raw_text
0,23,the company’s total net sales were billion and...,383.3,97.0,2023.0,The Company’s total net sales were $383.3 bill...
17,38,the following table shows the computation of b...,2023.0,2022.0,2021.0,The following table shows the computation of b...


In [12]:
metrics_df[
    metrics_df["metric_name"].str.contains("research and development", case=False, na=False)
]

,page,metric_name,2023,2022,2021,raw_text


In [14]:
def is_real_financial_table_row(text):
    text = str(text)

    numbers = re.findall(r"\(?\$?\s?\d{1,3}(?:,\d{3})+(?:\.\d+)?\)?", text)

    return len(numbers) >= 3

In [15]:
table_like_lines_df = lines_df[
    lines_df["text"].apply(is_real_financial_table_row)
].copy()

table_like_lines_df[["page", "text"]].head(50)

,page,text


In [16]:
output_path = Path("D:/sudhendra/learning projects/GraphRAG/data/apple_2023_pymupdf_financial_lines.csv")

metrics_df.to_csv(output_path, index=False)

output_path

WindowsPath('D:/sudhendra/learning projects/GraphRAG/data/apple_2023_pymupdf_financial_lines.csv')

trying again as upper failed

In [17]:
import fitz
import pandas as pd
import re
from pathlib import Path

In [ ]:
pdf_path = "D:/sudhendra/learning projects/GraphRAG/data/Apple23 report.pdf"
output_path = "D:/sudhendra/learning projects/GraphRAG/data/apple_2023_financial_table_rows_clean.csv"

In [31]:
doc = fitz.open(pdf_path)

all_lines = []

for page_index, page in enumerate(doc):
    lines = page.get_text("text").splitlines()

    for line_index, line in enumerate(lines):
        clean_line = line.replace("\xa0", " ").strip()

        if clean_line:
            all_lines.append({
                "page": page_index + 1,
                "line_index": line_index,
                "text": clean_line
            })

lines_df = pd.DataFrame(all_lines)

lines_df.head(30)

,page,line_index,text
0,1,0,UNITED STATES
1,1,1,SECURITIES AND EXCHANGE COMMISSION
2,1,2,"Washington, D.C. 20549"
3,1,3,FORM 10-K
4,1,4,(Mark One)
5,1,5,☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 1...
6,1,6,"For the fiscal year ended September 30, 2023"
7,1,7,or
8,1,8,☐ TRANSITION REPORT PURSUANT TO SECTION 13 ...
9,1,9,For the transition period from to...


In [32]:
def clean_number(value):
    value = str(value).strip()

    negative = False

    if "(" in value and ")" in value:
        negative = True

    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.replace("(", "")
    value = value.replace(")", "")
    value = value.strip()

    try:
        num = float(value)
        return -num if negative else num
    except:
        return None

In [33]:
def is_number_line(text):
    text = str(text).strip()

    if "%" in text:
        return False

    if text in ["$", "$ "]:
        return False

    # skip year headers like 2023, 2022, 2021
    if text in ["2023", "2022", "2021"]:
        return False

    pattern = r"^\(?\$?\s?\d{1,3}(?:,\d{3})*(?:\.\d+)?\)?$"

    return bool(re.match(pattern, text))

In [34]:
def is_possible_label(text):
    text = str(text).strip()
    lower = text.lower()

    if len(text) > 80:
        return False

    if not re.search(r"[a-zA-Z]", text):
        return False

    bad_phrases = [
        "apple inc.",
        "form 10-k",
        "see accompanying",
        "the following table",
        "years ended",
        "september",
        "change",
        "in millions",
        "dollars in millions",
        "net sales by category",
        "net sales by reportable segment",
        "cost of sales:",
        "net sales:",
        "operating expenses:",
        "earnings per share:",
        "shares used",
        "numerator:",
        "denominator:",
        "note ",
        "item "
    ]

    if any(phrase in lower for phrase in bad_phrases):
        return False

    return True

In [35]:
rows = []

for page in sorted(lines_df["page"].unique()):
    page_lines = lines_df[lines_df["page"] == page].reset_index(drop=True)

    for i in range(len(page_lines)):
        label = page_lines.loc[i, "text"]

        if not is_possible_label(label):
            continue

        values = []

        # Look ahead after label
        for j in range(i + 1, min(i + 12, len(page_lines))):
            candidate = page_lines.loc[j, "text"]

            if is_number_line(candidate):
                num = clean_number(candidate)
                if num is not None:
                    values.append(num)

            if len(values) == 3:
                break

        if len(values) == 3:
            rows.append({
                "page": page,
                "metric_name": label.lower(),
                "2023": values[0],
                "2022": values[1],
                "2021": values[2],
                "raw_label": label,
                "raw_values": values
            })

financial_rows_df = pd.DataFrame(rows)

financial_rows_df

,page,metric_name,2023,2022,2021,raw_label,raw_values
0,3,table of contents,1.0,5.00,16.0,TABLE OF CONTENTS,"[1.0, 5.0, 16.0]"
1,3,page,1.0,5.00,16.0,Page,"[1.0, 5.0, 16.0]"
2,3,part i,1.0,5.00,16.0,Part I,"[1.0, 5.0, 16.0]"
3,3,business,1.0,5.00,16.0,Business,"[1.0, 5.0, 16.0]"
4,3,risk factors,5.0,16.00,16.0,Risk Factors,"[5.0, 16.0, 16.0]"
...,...,...,...,...,...,...,...
488,58,s-3,4.1,4.28,4.1,S-3,"[4.1, 4.28, 4.1]"
489,77,exhibit 23.1,-1.0,-2.00,-3.0,Exhibit 23.1,"[-1.0, -2.0, -3.0]"
490,77,consent of independent registered public accou...,-1.0,-2.00,-3.0,Consent of Independent Registered Public Accou...,"[-1.0, -2.0, -3.0]"
491,77,"2022 employee stock plan,",-4.0,-5.00,-6.0,"2022 Employee Stock Plan,","[-4.0, -5.0, -6.0]"


In [23]:
structured_rows = []

for _, row in lines_df.iterrows():
    text = str(row["text"])

    text_lower = text.lower()

    has_keyword = any(keyword in text_lower for keyword in financial_keywords)

    numbers = extract_large_numbers(text)

    # Real financial table rows usually have 3 year values
    if has_keyword and len(numbers) >= 3:
        metric_name = remove_numbers_from_text(text)

        structured_rows.append({
            "page": int(row["page"]),
            "metric_name": metric_name,
            "2023": numbers[0],
            "2022": numbers[1],
            "2021": numbers[2],
            "raw_text": text
        })

financial_tables_df = pd.DataFrame(structured_rows)

financial_tables_df

,page,metric_name,2023,2022,2021,raw_text
0,24,the following table shows net sales by reporta...,2023.0,2022.0,2021.0,The following table shows net sales by reporta...
1,24,europe net sales decreased 1% or million durin...,824.0,2023.0,2022.0,Europe net sales decreased 1% or $824 million ...
2,24,rest of asia pacific net sales increased 1% or...,240.0,2023.0,2022.0,Rest of Asia Pacific net sales increased 1% or...
3,25,the following table shows net sales by categor...,2023.0,2022.0,2021.0,The following table shows net sales by categor...
4,26,products and services gross margin and gross m...,2023.0,2022.0,2021.0,Products and Services gross margin and gross m...
5,38,net sales disaggregated by significant product...,2023.0,2022.0,2021.0,Net sales disaggregated by significant product...
6,38,the following table shows the computation of b...,2023.0,2022.0,2021.0,The following table shows the computation of b...
7,51,". net sales for, and and long-lived assets as ...",2021.0,2023.0,2022.0,"2021. Net sales for 2023, 2022 and 2021 and lo..."


In [36]:
useful_keywords = [
    "iphone",
    "mac",
    "ipad",
    "wearables",
    "services",
    "products",
    "total net sales",
    "total cost of sales",
    "gross margin",
    "research and development",
    "selling, general and administrative",
    "total operating expenses",
    "operating income",
    "income before provision",
    "provision for income taxes",
    "net income",
    "basic earnings per share",
    "diluted earnings per share",
    "total assets",
    "total liabilities",
    "cash and cash equivalents"
]

In [37]:
clean_financial_rows_df = financial_rows_df[
    financial_rows_df["metric_name"].apply(
        lambda x: any(keyword in x for keyword in useful_keywords)
    )
].copy()

clean_financial_rows_df.reset_index(drop=True, inplace=True)

clean_financial_rows_df

,page,metric_name,2023,2022,2021,raw_label,raw_values
0,3,principal accountant fees and services,53.0,54.0,57.0,Principal Accountant Fees and Services,"[53.0, 54.0, 57.0]"
1,24,total net sales,383285.0,394328.0,365817.0,Total net sales,"[383285.0, 394328.0, 365817.0]"
2,25,iphone (1),200583.0,205489.0,191973.0,iPhone (1),"[200583.0, 205489.0, 191973.0]"
3,25,mac (1),29357.0,40177.0,35190.0,Mac (1),"[29357.0, 40177.0, 35190.0]"
4,25,ipad (1),28300.0,29292.0,31862.0,iPad (1),"[28300.0, 29292.0, 31862.0]"
...,...,...,...,...,...,...,...
57,50,operating income,12066.0,11569.0,9817.0,Operating income,"[12066.0, 11569.0, 9817.0]"
58,50,segment operating income,150888.0,152895.0,137006.0,Segment operating income,"[150888.0, 152895.0, 137006.0]"
59,50,research and development expense,-29915.0,-26251.0,-21914.0,Research and development expense,"[-29915.0, -26251.0, -21914.0]"
60,50,total operating income,114301.0,119437.0,108949.0,Total operating income,"[114301.0, 119437.0, 108949.0]"


In [38]:
clean_financial_rows_df[
    clean_financial_rows_df["metric_name"].str.contains("total net sales", case=False, na=False)
]

,page,metric_name,2023,2022,2021,raw_label,raw_values
1,24,total net sales,383285.0,394328.0,365817.0,Total net sales,"[383285.0, 394328.0, 365817.0]"
7,25,total net sales,383285.0,394328.0,365817.0,Total net sales,"[383285.0, 394328.0, 365817.0]"
13,26,percentage of total net sales,24932.0,25094.0,21973.0,Percentage of total net sales,"[24932.0, 25094.0, 21973.0]"
15,26,percentage of total net sales,54847.0,51345.0,43887.0,Percentage of total net sales,"[54847.0, 51345.0, 43887.0]"
20,31,total net sales,383285.0,394328.0,365817.0,Total net sales,"[383285.0, 394328.0, 365817.0]"
44,38,total net sales,383285.0,394328.0,365817.0,Total net sales,"[383285.0, 394328.0, 365817.0]"
61,51,total net sales,383285.0,394328.0,365817.0,Total net sales,"[383285.0, 394328.0, 365817.0]"


In [39]:
clean_financial_rows_df[
    clean_financial_rows_df["metric_name"].str.contains("research and development", case=False, na=False)
]

,page,metric_name,2023,2022,2021,raw_label,raw_values
12,26,research and development,29915.0,26251.0,21914.0,Research and development,"[29915.0, 26251.0, 21914.0]"
25,31,research and development,29915.0,26251.0,21914.0,Research and development,"[29915.0, 26251.0, 21914.0]"
50,43,"research and development credit, net",-1212.0,-1153.0,-1033.0,"Research and development credit, net","[-1212.0, -1153.0, -1033.0]"
52,44,capitalized research and development,6294.0,1267.0,4571.0,Capitalized research and development,"[6294.0, 1267.0, 4571.0]"
59,50,research and development expense,-29915.0,-26251.0,-21914.0,Research and development expense,"[-29915.0, -26251.0, -21914.0]"


In [40]:
clean_financial_rows_df[
    clean_financial_rows_df["metric_name"].str.contains("net income", case=False, na=False)
]

,page,metric_name,2023,2022,2021,raw_label,raw_values
31,31,net income,96995.0,99803.0,94680.0,Net income,"[96995.0, 99803.0, 94680.0]"
32,32,net income,96995.0,99803.0,94680.0,Net income,"[96995.0, 99803.0, 94680.0]"
36,34,net income,96995.0,99803.0,94680.0,Net income,"[96995.0, 99803.0, 94680.0]"
37,35,net income,96995.0,99803.0,94680.0,Net income,"[96995.0, 99803.0, 94680.0]"
38,35,adjustments to reconcile net income to cash ge...,11519.0,11104.0,11284.0,Adjustments to reconcile net income to cash ge...,"[11519.0, 11104.0, 11284.0]"
45,38,net income,96995.0,99803.0,94680.0,Net income,"[96995.0, 99803.0, 94680.0]"


In [41]:
clean_financial_rows_df.to_csv(output_path, index=False)

output_path

WindowsPath('D:/sudhendra/learning projects/GraphRAG/data/apple_2023_pymupdf_financial_lines.csv')

into JSON now


In [48]:
import pandas as pd
import json
from pathlib import Path

In [49]:
csv_path = "D:/sudhendra/learning projects/GraphRAG/data/apple_2023_pymupdf_financial_lines.csv"

df = pd.read_csv(csv_path)

df.head()

,page,metric_name,2023,2022,2021,raw_label,raw_values
0,3,principal accountant fees and services,53.0,54.0,57.0,Principal Accountant Fees and Services,"[53.0, 54.0, 57.0]"
1,24,total net sales,383285.0,394328.0,365817.0,Total net sales,"[383285.0, 394328.0, 365817.0]"
2,25,iphone (1),200583.0,205489.0,191973.0,iPhone (1),"[200583.0, 205489.0, 191973.0]"
3,25,mac (1),29357.0,40177.0,35190.0,Mac (1),"[29357.0, 40177.0, 35190.0]"
4,25,ipad (1),28300.0,29292.0,31862.0,iPad (1),"[28300.0, 29292.0, 31862.0]"


In [50]:
df["metric_name_clean"] = (
    df["metric_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [51]:
metric_mapping = {
    "total net sales": "total_net_sales",
    "gross margin": "gross_margin",
    "total gross margin": "gross_margin",
    "research and development": "research_and_development",
    "selling, general and administrative": "selling_general_and_administrative",
    "total operating expenses": "total_operating_expenses",
    "operating income": "operating_income",
    "net income": "net_income",
    "cash and cash equivalents": "cash_and_cash_equivalents",
    "total assets": "total_assets",
    "total liabilities": "total_liabilities",
    "basic": "basic_eps",
    "diluted": "diluted_eps"
}

In [52]:
financial_metrics = {}

for _, row in df.iterrows():
    metric_name = row["metric_name_clean"]

    if metric_name not in metric_mapping:
        continue

    clean_key = metric_mapping[metric_name]

    financial_metrics[clean_key] = {
        "label": metric_name,
        "values": {
            "2023": float(row["2023"]),
            "2022": float(row["2022"]),
            "2021": float(row["2021"])
        },
        "source_page": int(row["page"]),
        "source_type": "pymupdf_financial_line",
        "raw_text": row.get("raw_text", "")
    }

financial_metrics

{'total_net_sales': {'label': 'total net sales',
  'values': {'2023': 383285.0, '2022': 394328.0, '2021': 365817.0},
  'source_page': 51,
  'source_type': 'pymupdf_financial_line',
  'raw_text': ''},
 'gross_margin': {'label': 'gross margin',
  'values': {'2023': 169148.0, '2022': 170782.0, '2021': 152836.0},
  'source_page': 31,
  'source_type': 'pymupdf_financial_line',
  'raw_text': ''},
 'research_and_development': {'label': 'research and development',
  'values': {'2023': 29915.0, '2022': 26251.0, '2021': 21914.0},
  'source_page': 31,
  'source_type': 'pymupdf_financial_line',
  'raw_text': ''},
 'selling_general_and_administrative': {'label': 'selling, general and administrative',
  'values': {'2023': 24932.0, '2022': 25094.0, '2021': 21973.0},
  'source_page': 31,
  'source_type': 'pymupdf_financial_line',
  'raw_text': ''},
 'total_operating_expenses': {'label': 'total operating expenses',
  'values': {'2023': 54847.0, '2022': 51345.0, '2021': 43887.0},
  'source_page': 31,
  

In [53]:
output_path = Path("D:/sudhendra/learning projects/GraphRAG/data/apple_2023_financial_metrics.json")

with open(output_path, "w") as f:
    json.dump(financial_metrics, f, indent=4)

output_path

WindowsPath('D:/sudhendra/learning projects/GraphRAG/data/apple_2023_financial_metrics.json')

In [54]:
with open(output_path, "r") as f:
    metrics = json.load(f)

metrics.keys()

dict_keys(['total_net_sales', 'gross_margin', 'research_and_development', 'selling_general_and_administrative', 'total_operating_expenses', 'operating_income', 'net_income', 'cash_and_cash_equivalents', 'total_assets', 'total_liabilities'])

In [55]:
metrics["total_net_sales"]

{'label': 'total net sales',
 'values': {'2023': 383285.0, '2022': 394328.0, '2021': 365817.0},
 'source_page': 51,
 'source_type': 'pymupdf_financial_line',
 'raw_text': ''}

fucking finally man

In [56]:
with open("../data/apple_2023_financial_metrics.json","r") as f:
    metrics = json.load(f)

metrics

{'total_net_sales': {'label': 'total net sales',
  'values': {'2023': 383285.0, '2022': 394328.0, '2021': 365817.0},
  'source_page': 51,
  'source_type': 'pymupdf_financial_line',
  'raw_text': ''},
 'gross_margin': {'label': 'gross margin',
  'values': {'2023': 169148.0, '2022': 170782.0, '2021': 152836.0},
  'source_page': 31,
  'source_type': 'pymupdf_financial_line',
  'raw_text': ''},
 'research_and_development': {'label': 'research and development',
  'values': {'2023': 29915.0, '2022': 26251.0, '2021': 21914.0},
  'source_page': 31,
  'source_type': 'pymupdf_financial_line',
  'raw_text': ''},
 'selling_general_and_administrative': {'label': 'selling, general and administrative',
  'values': {'2023': 24932.0, '2022': 25094.0, '2021': 21973.0},
  'source_page': 31,
  'source_type': 'pymupdf_financial_line',
  'raw_text': ''},
 'total_operating_expenses': {'label': 'total operating expenses',
  'values': {'2023': 54847.0, '2022': 51345.0, '2021': 43887.0},
  'source_page': 31,
  